In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.ticker as ticker

from pathlib import Path
from itertools import product

In [ ]:
data_path = Path(os.getcwd()) / "Data-processing/Results/Omar/"
out_graph_path = Path(os.getcwd()) / "Data-processing/Results/Omar/graphs"
out_graph_path.mkdir(parent=True, exist_ok=True)
out_table_path = Path(os.getcwd()) / "Data-processing/Results/Omar/tables"
out_table_path.mkdir(parent=True, exist_ok=True)

data = pd.read_csv(data_path / "01-data-damBreakNoVelocity.csv")
time = pd.read_csv(data_path / "02-time-damBreakNoVelocity.csv")
profile = pd.read_csv(data_path / "03-u_profile-damBreakNoVelocity.csv", low_memory=False)
error = pd.read_csv(data_path / "04-error-damBreakNoVelocity.csv")

# Tools

In [3]:
index_columns = set(time.columns[0:-1])
quantity_columns = [
            set(data.columns) - index_columns, 
            set(time.columns) - index_columns,
            set(error.columns) - index_columns
        ]
index_columns = list(index_columns)
quantity_columns = [sorted(list(i)) for i in quantity_columns]

def marker_list_func(palette=None):
    return [".", "s", "o", "*"]

def color_list_func(palette=None):
    color_blind_colors = ['#56B4E9', '#D55E00', '#009E73', '#CC79A7', '#E69F00']
    return ['#ffd700', '#0000ff', '#ea5f94', '#ffb14e', '#fa8775','#009E73']
def color_map_func(quantity_list: list, palette: str = None): 

    color_list = color_list_func()

    if len(color_list) < len(quantity_list):
        print("WARNING: Color palette is smaller than quantity to represent. Repeated colours will be used.")
    
    return {quant: color_list[i % len(color_list)] for i, quant in enumerate(quantity_list)}

# Numerical convergence

#### Functions

##### Main plot

In [ ]:
def legend_plot_convergence(fig, color_map: dict):
    """ 
    Creates a single legend where it specifies:
        - What marker is associated to L2 norm and LInf
        - What color is linked to what resolution
    """
    # marker_handles = []
    # for error_type, marker in marker_map.items():
    #     handle = mlines.Line2D( [], [],
    #                             marker='.',
    #                             markersize=8, color='black',
    #                             label=error_type.strip()
    #                         )
    #     marker_handles.append(handle)
    color_handles = []
    for res, color in color_map.items():
        handle = mpatches.Patch(color=color, label=f"Res = {int(res)}")
        color_handles.append(handle)

    # A blank line as a spacer
    blank = mlines.Line2D([], [], color='none', label='')  

    fig.legend(
        handles=[
            # mlines.Line2D([], [], color='none', label='Error type'),   # header
            # *marker_handles,
            # blank,                                                       
            mlines.Line2D([], [], color='none', label='Resolution'),    # header
            *color_handles,
        ],
        loc='center left',    # places legend to the right of all subplots
        bbox_to_anchor = (1.0, 0.5),
        frameon=True,
        fontsize=11,
        handlelength=2.5,
        # title=f"Case for plant height {height}\n       and density {density}",
        # title_fontsize="medium"
    )

def plot_error_convergence(df: pd.DataFrame, error_type: str = "L2", ax_type: str = "linear"):

    fig, axs = plt.subplots(1, 3, sharey=True)

    resolutions = df["Resolution"].unique()
    orders = df["Order"].unique()

    color_map = color_map_func(resolutions)

    for resolution in resolutions:

        mask = (df["Resolution"] == resolution) & (df["Error Type"] == error_type)

        axs[0].plot(df.loc[mask,"Order"], df.loc[mask,"Water Height"],
                    linestyle='-',
                    color=color_map[resolution],
                    marker=".")

        axs[1].plot(df.loc[mask,"Order"], df.loc[mask, "Average Velocity"],
                    linestyle='-',
                    color=color_map[resolution],
                    marker=".")

        axs[2].errorbar(df.loc[mask,"Order"], df.loc[mask,"U profile"],
                        yerr=df.loc[mask, "std U profile"],
                        color=color_map[resolution], 
                        marker=".")

        axs[0].set_title("Water Height")
        axs[1].set_title("Average velocity")
        axs[2].set_title("Velocity profile")
        axs[0].set_ylabel("Error (a.u.)")
        axs[1].set_xlabel("Order of the polynomial")

    for ax in axs.flatten():
        ax.grid()
        ax.grid(which="minor", color="0.9")
        ax.set_xticks(range(orders.min(), orders.max()+1))
        if ax_type == "log":
            ax.set_yscale("log")
            ax.set_ylim(2e-6,7e-1) 
    fig.suptitle(
                f"""Error ({error_type} norm). Effects of polynomial order and grid resolution. 
                                Plant height {height} and Plant density {density}""", 
                fontsize=11, fontweight='bold'
            )
    
    return fig, axs, color_map

##### Percentage relative change

In [48]:
def plot_percentage_change_convergence(df: pd.DataFrame, 
                                        error_type: str = "L2", 
                                        ax_type: str = "linear",

                                    ):

    pct_df = compute_percentage_change(df, error_type=error_type)
    
    fig, axs = plt.subplots(1, 3, sharey=True)

    resolutions = pct_df["Resolution"].unique()
    orders = np.sort(pct_df["Order"].unique())
    color_map = color_map_func(resolutions)

    for resolution in resolutions:
        mask = pct_df["Resolution"] == resolution

        axs[0].plot(pct_df.loc[mask, "Order"], -1*pct_df.loc[mask, "Water Height"],
                    linestyle='-', color=color_map[resolution], marker=".")
        axs[1].plot(pct_df.loc[mask, "Order"], -1*pct_df.loc[mask, "Average Velocity"],
                    linestyle='-', color=color_map[resolution], marker=".")
        axs[2].plot(pct_df.loc[mask, "Order"], -1*pct_df.loc[mask, "U profile"],
                    linestyle='-', color=color_map[resolution], marker=".")

    axs[0].set_title("Water Height")
    axs[1].set_title("Average velocity")
    axs[2].set_title("Velocity profile")
    axs[0].set_ylabel("Relative error reduction (% compared to previous order)")
    axs[1].set_xlabel("Order of the polynomial")

    for ax in axs.flatten():
        ax.grid()
        ax.grid(which="minor", color="0.9")
        ax.set_xticks(range(orders.min(), orders.max() + 1))
        if ax_type == "log":
            ax.set_yscale("log")

    fig.suptitle(
        f"""Percentage in error reduction ({error_type} norm) compared to previous order. 
                        Plant height {height} and Plant density {density}""",
        fontsize=11, fontweight='bold'
    )
    return fig, axs, color_map

def compute_percentage_change(df: pd.DataFrame,
                               error_type: str,
                               group_cols: list = ["Resolution"],
                               quantities: list = ["Water Height", "Average Velocity", "U profile"]
                               ) -> pd.DataFrame:
    """
    Computes (new - old) / new * 100 between consecutive polynomial orders,
    for each group in `group_cols`, for a single error_type.

    Returns a long-format DataFrame with columns:
        group_cols + ["Order"] + quantities
    where "Order" is the *new* order in each comparison (old = Order - 1's
    neighbor in the sorted, available order list -- not literally Order-1,
    in case some orders are missing).
    """
    sub = df.loc[df["Error Type"] == error_type]

    records = []
    for keys, group in sub.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        group = group.sort_values("Order")

        orders = group["Order"].to_numpy()
        values = group[quantities].to_numpy()

        for i in range(1, len(orders)):
            old_vals, new_vals = values[i - 1], values[i]
            pct = (new_vals - old_vals) / old_vals * 100

            record = dict(zip(group_cols, keys))
            record["Order"] = orders[i]
            record.update(dict(zip(quantities, pct)))
            records.append(record)

    return pd.DataFrame(records)



##### Percentage cummulative

In [46]:
def plot_percentage_cumulative_convergence(df: pd.DataFrame, 
                                        error_type: str = "L2", 
                                        baseline_order: int = 0,
                                    ):

    pct_df = compute_cumulative_improvement(df, error_type=error_type)
    
    fig, axs = plt.subplots(1, 3, sharey=True)

    resolutions = pct_df["Resolution"].unique()
    orders = np.sort(pct_df["Order"].unique())
    color_map = color_map_func(resolutions)

    bar_width = 0.8 / len(resolutions)

    for i, resolution in enumerate(resolutions):
        mask = pct_df["Resolution"] == resolution
        group = pct_df.loc[mask].sort_values("Order")

        # offset each resolution's bars so they sit side-by-side per order
        x_positions = group["Order"].to_numpy() + (i - (len(resolutions) - 1) / 2) * bar_width

        axs[0].bar(x_positions, -1*group["Water Height"], width=bar_width,
                   color=color_map[resolution], label=resolution)
        axs[1].bar(x_positions, -1*group["Average Velocity"], width=bar_width,
                   color=color_map[resolution])
        axs[2].bar(x_positions, -1*group["U profile"], width=bar_width,
                   color=color_map[resolution])

    axs[0].set_title("Water Height")
    axs[1].set_title("Average velocity")
    axs[2].set_title("Velocity profile")
    axs[0].set_ylabel(f"% error reduction compared to depth averaging (order 0)")
    axs[1].set_xlabel("Order of the polynomial")

    for ax in axs.flatten():
        ax.set_xlim(0.2,5.8)
        ax.set_ylim(45,105) if error_type == "L2" else None
        ax.set_xticks(range(1, 6))


    fig.suptitle(
        f"""% Error reduction compared to depth averaged soluton ({error_type} norm) 
                    Plant height {height} and Plant density {density}""",
        fontsize=11, fontweight='bold'
    )
    return fig, axs, color_map
def compute_cumulative_improvement(df: pd.DataFrame,
                                    error_type: str = "L2",
                                    baseline_order: int = 0,
                                    group_cols: list = ["Resolution"],
                                    quantities: list = ["Water Height", "Average Velocity", "U profile"]
                                ) -> pd.DataFrame:
    """
    # For each group, computes percentage improvement of every order
    # relative to a fixed baseline order (default: order 0):
    #     (baseline - value) / baseline * 100
    # Positive = improvement over the baseline. Baseline order itself
    # is included with 0% by definition (and can be dropped by the caller
    # if not needed).
    """
    sub = df.loc[df["Error Type"] == error_type]

    records = []
    for keys, group in sub.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        group = group.sort_values("Order")

        baseline_row = group.loc[group["Order"] == baseline_order]
        if baseline_row.empty:
            continue  # this group has no order-0 case to compare against

        baseline_vals = baseline_row[quantities].to_numpy()[0]

        for _, row in group.iterrows():
            vals = row[quantities].to_numpy()
            improvement = (vals - baseline_vals) / baseline_vals * 100

            record = dict(zip(group_cols, keys))
            record["Order"] = row["Order"]
            record.update(dict(zip(quantities, improvement)))
            records.append(record)

    return pd.DataFrame(records)

##### Time

In [ ]:
def plot_time_convergence(df: pd.DataFrame, resolution:int, height: float):

    resolutions = pct_df["Resolution"].unique()
    orders = np.sort(pct_df["Order"].unique())
    color_map = color_map_func(resolutions)

    fig, axs = plt.subplots(1)

    for resolution in resolutions:
        mask = pct_df["Resolution"] == resolution
        group = pct_df.loc[mask].sort_values("Order")

        axs.plot(df.loc[mask,"Order"], df.loc[mask,"Time"],
                   color=color_map[resolution], label=resolution)
        axs[1].bar(x_positions, -1*group["Average Velocity"], width=bar_width,
                   color=color_map[resolution])
        axs[2].bar(x_positions, -1*group["U profile"], width=bar_width,
                   color=color_map[resolution])

    axs.set_title("Time vs Order")
    axs.set_ylabel(f"Time (s)")
    axs.set_xlabel("Order of the polynomial")

    fig.suptitle("""        Computation time for each order
                    Plant height {height} and Plant density {density}""",
                    fontsize=11, fontweight='bold'
    )

    return fig, axs, color_map

##### Defunct

In [38]:
def find_sufficient_order(pct_df: pd.DataFrame,
                           tolerance: float = 1.0,
                           quantities: list = ["Water Height", "Average Velocity", "U profile"],
                           group_cols: list = ["Resolution"]) -> pd.DataFrame:
    """
    For each group in group_cols, finds the smallest Order at which
    |percentage change| stays below `tolerance` (%) for ALL quantities,
    for that order and every subsequent order present in the data.

    Returns a DataFrame with columns group_cols + ["Sufficient Order"].
    Rows where no order satisfies this are marked with NaN.
    """
    # worst-case (max abs) percentage change across quantities, per row
    worst = pct_df.copy()
    worst["Max Abs Pct Change"] = worst[quantities].abs().max(axis=1)

    records = []
    for keys, group in worst.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        group = group.sort_values("Order")

        below = group["Max Abs Pct Change"].to_numpy() < tolerance
        orders = group["Order"].to_numpy()

        sufficient_order = np.nan
        # find first index i such that below[i:] is all True (monotone "stays under")
        for i in range(len(below)):
            if below[i:].all():
                sufficient_order = orders[i]
                break

        record = dict(zip(group_cols, keys))
        record["Sufficient Order"] = sufficient_order
        records.append(record)

    return pd.DataFrame(records)

#### Plot

In [61]:
%%capture

SAVE_FIG = True

# Simulations for convergence have resolution up to 1e4
# Find those
mask_res = error["Resolution"] == 10000
idx_res = error.loc[mask_res].index

heights = error.loc[idx_res, "Plant Height"].unique()
densities = error.loc[idx_res, "Plant Density"].unique()
mask_h = error["Plant Height"].isin(heights)
mask_dens = error["Plant Density"].isin(densities)

convergence_arr = error.loc[(mask_h) & (mask_dens)]

for density, height in product(densities, heights):
    mask = (convergence_arr["Plant Height"]==height) & (convergence_arr["Plant Density"]==density)
    
    errors_available = convergence_arr['Error Type'].unique()

    for error_type in errors_available:
        # Error in absolute terms

        fig_error, axs_error, cm_error = plot_error_convergence(df=convergence_arr.loc[mask], error_type=error_type)
        legend_plot_convergence(fig_error, color_map=cm_error)
        fig_error.tight_layout()

        #Percentage of decrease per step
        fig_rel_err, axs_rel_err, cm_rel_err = plot_percentage_change_convergence(df=convergence_arr.loc[mask], error_type=error_type)
        legend_plot_convergence(fig_rel_err, color_map=cm_rel_err)
        fig_rel_err.tight_layout()

        fig_cum_err, axs_cum_err, cm_cum_err = plot_percentage_cumulative_convergence(df=convergence_arr.loc[mask], error_type=error_type)
        legend_plot_convergence(fig_cum_err, color_map=cm_cum_err)
        fig_cum_err.tight_layout()
        
        if SAVE_FIG == True:
            fig_error.savefig(fname=(out_graph_path / "Plot_convergence_{error_type}_error.png"), format="png", dpi=600)
            fig_rel_err.savefig(fname=(out_graph_path / "Plot_convergence_{error_type}_relative_error.png"), format="png", dpi=600)
            fig_error.savefig(fname=(out_graph_path / "Plot_convergence_{error_type}_cumulative_error.png"), format="png", dpi=600)


FileNotFoundError: [Errno 2] No such file or directory: '/home/fosser/Documents/Uni/y3.4-Thesis/code/SpatiallyAdaptiveMomentModels/Nonlinear-systems/Data-processing/Results/Omar/graphs/Plot_convergence_{error_type}_error.png'

# wait

In [ ]:


############################    All in one plot     ############################
fig, axs = plt.subplots(3, 5, figsize=(10,10), sharex=True, sharey=True)

for i, height in enumerate(plant_h_list):

    for resolution in resolution_list:

        mask = (errors["Plant Height"] == height) & (errors["Resolution"] == resolution)
        
        for error_type in error_types:

            axs[0,i].plot(errors.loc[mask,"Order"], errors.loc[mask, error_type+"Water Height"],
                        linestyle='-',
                        color=color_map[resolution],
                        marker=marker_map[error_type])

            axs[1,i].plot(errors.loc[mask,"Order"], errors.loc[mask, error_type+"Average Velocity"],
                        linestyle='-',
                        color=color_map[resolution],
                        marker=marker_map[error_type])

            axs[2,i].errorbar(errors.loc[mask,"Order"], errors.loc[mask, error_type+"U profile"],
                            yerr=errors.loc[mask, "L2 std U profile"],
                            color=color_map[resolution], 
                            marker=marker_map[error_type])

for ax,h in zip(axs[0,:],plant_h_list): ax.set_title(f"Plant height {h}")
axs[0,0].set_ylabel("Error Water Height")
axs[1,0].set_ylabel("Error Avg velocity")
axs[2,0].set_ylabel("Error Velocity profile")
for ax in axs[-1,:]: ax.set_xlabel("Order")


for ax in axs.flatten():
    ax.set_yscale("log")
    ax.grid()
    ax.grid(which="minor", color="0.9")
    ax.set_ylim(2e-6,7e-1)
    ax.set_xticks(range(0,6))
create_legend_order_vs_error(fig)
fig.tight_layout()
plt.show()

In [ ]:
resolution_list = errors["Resolution"].unique()
plant_h_list = errors["Plant Height"].unique()        

# Error vs resolution
fig, axs = plt.subplots(3, 5, figsize=(20,20), sharex=True)

plt.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 12,
    'lines.linewidth': 2.0,
    'lines.markersize': 7,
})
error_types = ["L2 "]
for i, height in enumerate(plant_h_list):

    for resolution in resolution_list:

        mask = (errors["Plant Height"] == height) & (errors["Resolution"] == resolution)

        for j, error_type in enumerate(error_types):
            axs[0+3*j, i].plot(errors.loc[mask,"Order"], errors.loc[mask, error_type+"Water Height"],
                        linestyle='-',
                        marker=marker_map[error_type], 
                        color=color_map[resolution]
                    )

            axs[1+3*j, i].plot(errors.loc[mask,"Order"], errors.loc[mask, error_type+"Average Velocity"],
                        linestyle='-',
                        marker=marker_map[error_type], 
                        color=color_map[resolution]
                    )

            axs[2+3*j, i].errorbar(errors.loc[mask,"Order"], errors.loc[mask, error_type+"U profile"],
                        yerr=errors.loc[mask, "L2 std U profile"],
                        linestyle='-',
                        marker=marker_map[error_type], 
                        color=color_map[resolution]
                    )

for ax,h in zip(axs[0,:],plant_h_list): ax.set_title(f"Plant height {h}")
# for error_type in error_types:
#     for title in zip(error_types,quantity):
#TODO: I forgot what this was for hahaha

for ax, error_type, quantity in product(axs[:,0], error_types, quantities_list):
    ax.set_ylabel(error_type+quantity)
for ax in axs[-1,:]: ax.set_xlabel("Order")# of the polynomial")
for ax in axs[:3,:].flatten(): ax.set_ylim(0, 3.5e-3)
for ax in axs[3:,:].flatten(): ax.set_ylim(0, 4e-1)


for ax in axs.flatten():
    ax.grid()
    ax.grid(which="minor", color="0.9")
    ax.set_xticks(range(0,6))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: '{:.1e}'.format(x)))
create_legend_order_vs_error(fig)
fig.tight_layout()

### Order vs Time for each res

#### Individual

In [ ]:
fig, axs = plt.subplots(1)

for resolution in resolution_list:

    mask = (data_time["Plant Height"] == 1.0) & (data_time["Resolution"] == resolution)
    
    axs.plot(data_time.loc[mask,"Order"], data_time.loc[mask, "Time"] / 3600,
                linestyle='-',
                color=color_map[resolution],
                marker=marker_map[error_type],
                label=f"Resolution: {int(resolution)}")

axs.set_title("Order vs Computation Time")
axs.set_xlabel("Order")
axs.set_ylabel("Time (hours)")
axs.legend()
axs.grid()

fig.suptitle("Plant height 1.0", fontsize=11, fontweight='bold')
fig.tight_layout()

plt.show()

#### combined Individual and commulative

In [ ]:
fig, axs = plt.subplots(1,2)

cummulative_time = []
for resolution in resolution_list:

    mask = (data_time["Plant Height"] == 1.0) & (data_time["Resolution"] == resolution)
    
    axs[0].plot(data_time.loc[mask,"Order"], data_time.loc[mask, "Time"] / 3600,
                linestyle='-',
                color=color_map[resolution],
                marker=marker_map[error_type],
                label=f"Resolution: {int(resolution)}")

    cummulative_time.append(data_time.loc[mask, "Time"].sum() / 3600)

axs[1].plot(resolution_list, cummulative_time,
                linestyle='-',
                marker=".",
                color="black")

axs[0].set_title("Order vs Computation Time")
axs[0].set_xlabel("Order")
axs[0].set_ylabel("Time (hours)")

axs[1].set_title("Total Computation Time per resolution")
axs[1].set_xlabel('Resolution')
axs[1].set_ylabel("Cummulative Time (hours)")
axs[0].legend()
for ax in axs.flatten(): 
    ax.grid()

fig.suptitle("Plant height 1.0", fontsize=11, fontweight='bold')
fig.tight_layout()

plt.show()

### Dunno

In [ ]:
mask_3 = mask_func(velocity_profile, 3, 5000, height)
resolution=5000

velocity_profile.loc[mask_3,"0.0":].reset_index(drop=1).loc[resolution//2]

In [ ]:
def create_legend_velocity_profiles(fig):
    """ 
    Creates a single legend where it specifies:
        - What marker is associated to L2 norm and LInf
        - What color is linked to what resolution
    """
    marker_handles = []
    for order, marker in marker_map.items():
        handle = mlines.Line2D( [], [],
                                marker=marker,
                                markersize=8, color='black',
                                label=f"{int(order)}"
                            )
        marker_handles.append(handle)

    color_handles = []
    for height, color in color_map.items():
        handle = mpatches.Patch(color=color, label=f"Res = {int(res)}")
        color_handles.append(handle)

    # A blank line as a spacer
    blank = mlines.Line2D([], [], color='none', label='')  

    fig.legend(
        handles=[
            mlines.Line2D([], [], color='none', label='Order'),   # header
            *marker_handles,
            blank,                                                       
            mlines.Line2D([], [], color='none', label='O'),    # header
            *color_handles,
        ],
        loc='center left',    # places legend to the right of all subplots
        bbox_to_anchor = (1.0, 0.5),
        frameon=True,
        fontsize=11,
        handlelength=2.5,)

In [ ]:
def mask_func(array, order, resolution, height):
    return (array["Order"] == order) & (array["Resolution"]==resolution) & (array["Plant Height"]==height)

z_values = np.linspace(0,1,100)

fig, axs = plt.subplots(1,5, tight_layout=True, sharey=1)

for i,height in enumerate(plant_h_list):
    for order in range(6):
        mask = mask_func(velocity_profile, order, 5000, height)

        if i==4:
            axs[i].plot(velocity_profile.loc[mask,"0.0":].reset_index(drop=1).loc[5000//2], z_values,
                    label=f"{order}")
        else:
            axs[i].plot(velocity_profile.loc[mask,"0.0":].reset_index(drop=1).loc[5000//2], z_values)


fig.legend(title="Order",
        loc='center left',
        bbox_to_anchor = (1.0, 0.5),
        frameon=True,
        fontsize=11,
        handlelength=2.5)


In [ ]:
def mask_func(array, order, resolution, height):
    return (array["Order"] == order) & (array["Resolution"]==resolution) & (array["Plant Height"]==height)

z_values = np.linspace(0,1,100)

fig, axs = plt.subplots(1, tight_layout=True, sharey=1)

for order in range(7):
    mask = mask_func(velocity_profile, order, 5000, 1.0)

    axs.plot(velocity_profile.loc[mask,"0.0":].reset_index(drop=1).loc[5000//2], z_values,
                label=f"{order}")
axs.grid()
axs.set_xlabel("Velocity (m/s)")
axs.set_ylabel("Height")
fig.suptitle("Velocity profiles, plant height 1.0")
fig.legend(title="Order",
        loc='center left',
        bbox_to_anchor = (1.0, 0.5),
        frameon=True,
        fontsize=11,
        handlelength=2.5)

In [ ]:
def mask_func(array, order, resolution, height):
    return (array["Order"] == order) & (array["Resolution"]==resolution) & (array["Plant Height"]==height)

z_values = np.linspace(0,1,100)

fig, axs = plt.subplots(1,3, tight_layout=True, sharey=1)

color_map = {order: f"C{i}" for i, order in enumerate(range(3, 6))}
# C0, C1, C2 are matplotlib's default color cycle colors

for i,height in enumerate([0.0, 0.5, 1.0]):
    for order in range(3, 6):
        mask = mask_func(velocity_profile, order, 5000, height)
        label = str(order) if height == 1.0 else None
        axs[i-3].plot(
            velocity_profile.loc[mask, "0.0":].reset_index(drop=1).loc[5000//2],
            z_values,
            color=color_map[order],   # ← explicit color
            label=label
        )
line_x = np.linspace(0,1,10)
axs[0].plot(line_x, np.full_like(line_x, 0), 'r--')
axs[1].plot(line_x, np.full_like(line_x, 0.5), 'r--')
axs[2].plot(line_x, np.full_like(line_x, 1), 'r--')

for ax in axs: ax.set_xlim(0.05,0.46)
axs[0].set_title(f"Plant height: 0.0")
axs[1].set_title(f"Plant height: 0.5")
axs[2].set_title(f"Plant height: 1.0")
for ax in axs: ax.grid()
for ax in axs: ax.set_xlabel("Velocity (m/s)")
axs[0].set_ylabel("Height")
fig.suptitle("Some velocity profiles")
fig.legend(title="Order",
        loc='center left',
        bbox_to_anchor = (1.0, 0.5),
        frameon=True,
        fontsize=11,
        handlelength=2.5)

In [ ]:
Average error

# fig, axs = plt.subplots(1,5, tight_layout=True)
# for i, height in enumerate(plant_h_list):

#     for resolution in resolution_list:

#         mask = errors["Resolu"]
#         axs[i] = plt.plot(errors["Order"])

